In [9]:
!pip install -q \
    langchain \
    langchain-google-genai \
    langchain-qdrant \
    qdrant-client \
    datasets \
    pandas \
    langchain-text-splitters

In [3]:
from google.colab import userdata
api_key = userdata.get("GEMINI_API_KEY")

In [4]:
import os
import pandas as pd
from datasets import load_dataset

corpus_dataset = load_dataset(
    "rag-datasets/rag-mini-wikipedia",
    "text-corpus"
)

print(corpus_dataset)

README.md:   0%|          | 0.00/719 [00:00<?, ?B/s]

data/passages.parquet/part.0.parquet: reconstructing file:   0%|          |  0.00B /  797kB            

data/passages.parquet/part.0.parquet: downloading bytes:           |  0.00B            

Generating passages split:   0%|          | 0/3200 [00:00<?, ? examples/s]

DatasetDict({
    passages: Dataset({
        features: ['passage', 'id'],
        num_rows: 3200
    })
})


In [5]:

sampled_corpus = corpus_dataset["passages"].shuffle(seed=42).select(range(200))

print("Sample size:", len(sampled_corpus))
print("\nFirst passage:")
print(sampled_corpus[0])

Sample size: 200

First passage:
{'passage': 'Map of Uruguay', 'id': 28}


# Phase 2

In [6]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content=row["passage"],
        metadata={"id": row["id"]}
    )
    for row in sampled_corpus
]

print("Number of documents:", len(documents))
print("\nFirst document:")
print(documents[0])

Number of documents: 200

First document:
page_content='Map of Uruguay' metadata={'id': 28}


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))
print("\nFirst chunk:")
print(chunks[0].page_content)

Number of chunks: 283

First chunk:
Map of Uruguay


In [15]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    output_dimensionality=768,
    api_key=api_key
)

test_vector = embeddings.embed_query(chunks[0].page_content)

print("Embedding dimension:", len(test_vector))

Embedding dimension: 768


In [16]:
from qdrant_client import QdrantClient

qdrant_client = QdrantClient(":memory:")

print("Qdrant client created successfully!")

Qdrant client created successfully!


In [22]:
from qdrant_client.http.models import Distance, VectorParams

if qdrant_client.collection_exists("mini_wikipedia"):
    qdrant_client.delete_collection("mini_wikipedia")

qdrant_client.create_collection(
    collection_name="mini_wikipedia",
    vectors_config=VectorParams(
        size=768,
        distance=Distance.COSINE
    )
)

print("Collection created:", "mini_wikipedia")

Collection created: mini_wikipedia


In [23]:
import time

batch_size = 50

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]

    print(f"Embedding chunks {i + 1} to {i + len(batch)}...")

    vectorstore.add_documents(batch)

    if i + batch_size < len(chunks):
        print("Waiting 35 seconds...")
        time.sleep(35)

print("All chunks stored successfully!")

Embedding chunks 1 to 50...
Waiting 35 seconds...
Embedding chunks 51 to 100...
Waiting 35 seconds...
Embedding chunks 101 to 150...
Waiting 35 seconds...
Embedding chunks 151 to 200...
Waiting 35 seconds...
Embedding chunks 201 to 250...
Waiting 35 seconds...
Embedding chunks 251 to 283...
All chunks stored successfully!


# Phase 4: Retriever

In [24]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

print("Retriever created successfully!")

Retriever created successfully!


In [25]:
question = "What is Uruguay?"

retrieved_docs = retriever.invoke(question)

print("Number of retrieved chunks:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print(doc.page_content)

Number of retrieved chunks: 4

--- Retrieved Chunk 1 ---
Basketball, rugby union, and tennis are other popular sports in Uruguay.

--- Retrieved Chunk 2 ---
Map of Uruguay

--- Retrieved Chunk 3 ---
Many of the European immigrants arrived in Uruguay in the late 1800s and have heavily influenced the architecture and culture of Montevideo and other major cities. For this reason, Montevideo and life within the city are reminiscent of parts of Europe. For example Barcelona, Thessaloniki or Tel-Aviv are said to be similar to Montevideo in different aspects  /ref>

--- Retrieved Chunk 4 ---
Villas Miseria in Argentina, Barrios in Venezuela, Arrabales in Spain, Poblaciones Callampa in Chile or Jacales in Mexico.


In [26]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [36]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    api_key=api_key,
)

print("LLM ready!")

LLM ready!


In [37]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Answer ONLY from the context below in 1-2 short sentences. "
    "If the answer isn't there, say 'I don't know'.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

print("Prompt created successfully!")

Prompt created successfully!


In [38]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain created successfully!")

RAG chain created successfully!


In [39]:
example_question = "What sports are popular in Uruguay?"

example_docs = retriever.invoke(example_question)
example_answer = chain.invoke(example_question)

print("Question:", example_question)

print("\nRetrieved chunks:")
for i, doc in enumerate(example_docs, 1):
    print(f"\n--- Chunk {i} ---")
    print(doc.page_content)

print("\nFinal answer:")
print(example_answer)

Question: What sports are popular in Uruguay?

Retrieved chunks:

--- Chunk 1 ---
Basketball, rugby union, and tennis are other popular sports in Uruguay.

--- Chunk 2 ---
Many of the European immigrants arrived in Uruguay in the late 1800s and have heavily influenced the architecture and culture of Montevideo and other major cities. For this reason, Montevideo and life within the city are reminiscent of parts of Europe. For example Barcelona, Thessaloniki or Tel-Aviv are said to be similar to Montevideo in different aspects  /ref>

--- Chunk 3 ---
Football (soccer) is popular in Romania, the most internationally known player being Gheorghe Hagi, who played for Steaua BucureÅti (Romania), Real Madrid, FC Barcelona (Spain) and Galatasaray (Turkey), among others. In 1986, the Romanian soccer club Steaua BucureÅti became the first Eastern European club ever, and only one of the two (the other being Red Star Belgrade) to win the prestigious European Champions Cup title.In 1989, it played

In [40]:
from datasets import load_dataset

qa_dataset = load_dataset(
    "rag-datasets/rag-mini-wikipedia",
    "question-answer"
)

print(qa_dataset)

data/test.parquet/part.0.parquet: reconstructing file:   0%|          |  0.00B / 54.4kB            

data/test.parquet/part.0.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/918 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['question', 'answer', 'id'],
        num_rows: 918
    })
})


In [41]:
eval_dataset = qa_dataset["test"].shuffle(seed=42).select(range(20))

print("Evaluation samples:", len(eval_dataset))
print("\nFirst sample:")
print(eval_dataset[0])

Evaluation samples: 20

First sample:
{'question': "What did Cleveland's opponents say in 1884 to counter his innocent image?", 'answer': 'That he had fathered an illegitimate child', 'id': 711}


In [42]:
results = []

for i, row in enumerate(eval_dataset, start=1):
    question = row["question"]
    ground_truth = row["answer"]


    retrieved_docs = retriever.invoke(question)


    retrieved_chunks = [
        doc.page_content[:200]
        for doc in retrieved_docs
    ]


    prediction = chain.invoke(question)

    correct = int(
        ground_truth.lower() in prediction.lower()
    )

    results.append({
        "question": question,
        "retrieved_chunks": " || ".join(retrieved_chunks),
        "prediction": prediction,
        "ground_truth": ground_truth,
        "correct": correct
    })

    print(f"\n{'=' * 80}")
    print(f"Question {i}/20")
    print(f"{'=' * 80}")

    print("\nQuestion:")
    print(question)

    print("\nRetrieved chunks:")
    for j, chunk in enumerate(retrieved_chunks, start=1):
        print(f"\n--- Chunk {j} ---")
        print(chunk)

    print("\nPrediction:")
    print(prediction)

    print("\nGround truth:")
    print(ground_truth)

    print("\nCorrect:", correct)

accuracy = sum(row["correct"] for row in results) / 20

print(f"\n{'=' * 80}")
print(f"Final Accuracy: {accuracy:.2%}")
print(f"{'=' * 80}")


Question 1/20

Question:
What did Cleveland's opponents say in 1884 to counter his innocent image?

Retrieved chunks:

--- Chunk 1 ---
Cleveland was born in Caldwell, New Jersey to the Reverend Richard Cleveland and Anne Neal.  He was the fifth of nine children, five sons and four daughters.  He was named Stephen Grover in honor of t

--- Chunk 2 ---
* Cleveland, Grover.  about Hawaii.'' (1893).

--- Chunk 3 ---
In addition to the pardon dispute and lingering anti-Republican sentiment, Ford had to counter a plethora of negative media imagery. Chevy Chase often did pratfalls on Saturday Night Live, imitating F

--- Chunk 4 ---
Under the guise of a vacation cruise, Cleveland, accompanied by lead surgeon Dr. Joseph Bryant, left for New York. Bryant, joined by his assistants Dr. John F. Erdmann, Dr. W.W. Keen Jr., Dr. Ferdinan

Prediction:
I don't know.

Ground truth:
That he had fathered an illegitimate child

Correct: 0


GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [43]:
import time

# Keep results already completed
completed_results = results.copy()

# Continue from the next unanswered question
start_index = len(completed_results)

for i in range(start_index, len(eval_dataset)):
    row = eval_dataset[i]

    question = row["question"]
    ground_truth = row["answer"]

    print(f"\n{'=' * 80}")
    print(f"Question {i + 1}/20")
    print(f"{'=' * 80}")

    # Retrieve documents
    retrieved_docs = retriever.invoke(question)

    retrieved_chunks = [
        doc.page_content[:200]
        for doc in retrieved_docs
    ]

    print("\nQuestion:")
    print(question)

    print("\nRetrieved chunks:")
    for j, chunk in enumerate(retrieved_chunks, start=1):
        print(f"\n--- Chunk {j} ---")
        print(chunk)

    # Try the LLM, retrying temporary 503 errors
    while True:
        try:
            prediction = chain.invoke(question)
            break
        except Exception as e:
            if "503" in str(e) or "UNAVAILABLE" in str(e):
                print("\nGemini is temporarily unavailable. Waiting 30 seconds...")
                time.sleep(30)
            else:
                raise e

    correct = int(
        ground_truth.lower() in prediction.lower()
    )

    completed_results.append({
        "question": question,
        "retrieved_chunks": " || ".join(retrieved_chunks),
        "prediction": prediction,
        "ground_truth": ground_truth,
        "correct": correct
    })

    print("\nPrediction:")
    print(prediction)

    print("\nGround truth:")
    print(ground_truth)

    print("\nCorrect:", correct)

# Replace results with the complete evaluation
results = completed_results

accuracy = sum(row["correct"] for row in results) / len(results)

print(f"\n{'=' * 80}")
print(f"Completed: {len(results)}/20")
print(f"Final Accuracy: {accuracy:.2%}")
print(f"{'=' * 80}")


Question 2/20

Question:
Does otter give birth or lay egg?

Retrieved chunks:

--- Chunk 1 ---
Unlike most marine mammals such as (seals or whales), sea otters do not have a layer of insulating blubber. As with other species of otter, they rely on a layer of air trapped in their fur, which they

--- Chunk 2 ---
Polar bears mate in April/May over a one week period needed to induce ovulation. The fertilized egg then remains in a suspended state until August or September. During these 4 months, the females then

--- Chunk 3 ---
Eurasian otter

--- Chunk 4 ---
where food is available year-round, they may not enter a den until October. Cubs are born in December without awakening the mother. She remains dormant while nursing her cubs until the family emerges 

Gemini is temporarily unavailable. Waiting 30 seconds...

Gemini is temporarily unavailable. Waiting 30 seconds...

Prediction:
I don't know.

Ground truth:
give birth

Correct: 0

Question 3/20

Question:
How many days did it take th

In [47]:

eval_df = pd.DataFrame(results)

eval_df.to_csv("eval_results.csv", index=False)

print("Saved successfully!")
print("Rows:", len(eval_df))
print("Accuracy:", f"{eval_df['correct'].mean():.2%}")

Saved successfully!
Rows: 20
Accuracy: 15.00%
